In [ ]:
#####     31位志愿者总步数柱状图
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import seaborn as sns
# 设置中文字体（三种备选方案）
try:
    # 方案1：使用系统自带黑体
    plt.rcParams['font.sans-serif'] = ['SimHei']  
    
    # 解决负号显示问题
    plt.rcParams['axes.unicode_minus'] = False  
except Exception as e:
    print(f"字体设置失败，请手动安装中文字体。错误信息：{str(e)}")
def show_main_summary_chart(results_df):
    """显示所有志愿者的总步数汇总图"""
    plt.figure(figsize=(16, 8))
    
    # 按步数排序
    results_df.sort_values('日总步数', inplace=True)
    
    # 设置颜色
    colors = ['red' if x < 1000 else 'gold' if x < 5000 else 'green' 
             for x in results_df['日总步数']]
    
    # 创建柱状图
    bars = plt.bar(results_df['志愿者ID'], results_df['日总步数'], color=colors)
    
    # 设置图表标题和标签
    plt.title('31位志愿者24小时总步数比较', fontsize=16)
    plt.xlabel('志愿者ID', fontsize=14)
    plt.ylabel('24小时总步数', fontsize=14)
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(fontsize=10)
    plt.show()

def display_volunteer_data(folder_path):
    """显示所有志愿者数据图表"""
    results = {'志愿者ID': [], '日总步数': [], '分析数据': []}
    
    # 处理所有文件
    for file_name in sorted(os.listdir(folder_path)):
        if file_name.endswith('_steps.txt'):
            volunteer_id = file_name.split('_')[0]
            file_path = os.path.join(folder_path, file_name)
            
            print(f"正在处理: {volunteer_id}")
            daily_steps, daily_data = process_steps_file(file_path)
            
            if daily_steps is not None:
                results['志愿者ID'].append(volunteer_id)
                results['日总步数'].append(daily_steps)
                results['分析数据'].append(daily_data)
    
    # 创建DataFrame
    results_df = pd.DataFrame(results)
    
    if results_df.empty:
        print("没有找到有效数据")
        return
    
    # 1. 显示主汇总图
    show_main_summary_chart(results_df)
    
    # 2. 显示每位志愿者的细节图
    for _, row in results_df.iterrows():
        show_volunteer_details(row['志愿者ID'], row['分析数据'], row['日总步数'])
        
        # 询问是否继续显示下一个
        if not input("按Enter继续显示下一个志愿者，或输入q退出: ") == "":
            break

if __name__ == "__main__":
    data_folder = 'steps'
    display_volunteer_data(data_folder)

In [ ]:
def plot_combined_hourly_distribution(all_hourly_data):
    """绘制所有志愿者的步数时段分布总图"""
    plt.figure(figsize=(16, 8))
    
    # 为每条线设置不同的颜色
    colors = plt.cm.tab20(np.linspace(0, 1, len(all_hourly_data)))
    
    # 绘制每条折线
    for (vid, data), color in zip(all_hourly_data.items(), colors):
        plt.plot(data['hour'], data['steps'], 
                linestyle='-', linewidth=1.5, 
                color=color, label=vid)
    
    plt.title('所有志愿者24小时步数时段分布', fontsize=16)
    plt.xlabel('时间 (小时)', fontsize=12)
    plt.ylabel('步数', fontsize=12)
    
    # 设置x轴刻度
    plt.xticks(range(0, 24), [f'{h}:00' for h in range(24)], rotation=45)
    plt.xlim(-0.5, 23.5)
    
    # 添加图例(放在图表外部右侧)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', 
              borderaxespad=0., fontsize=8, title='志愿者ID')
    
    # 添加网格线
    plt.grid(True, linestyle='--', alpha=0.4)
    
    # 标记最活跃的3个时段
    avg_steps = pd.concat([df.set_index('hour') for df in all_hourly_data.values()], axis=1).mean(axis=1)
    top3_hours = avg_steps.nlargest(3).index
    for hour in top3_hours:
        plt.axvline(x=hour, color='red', linestyle=':', alpha=0.5)
        plt.text(hour, plt.ylim()[1]*0.95, f'{hour}:00', 
                ha='center', va='top', color='red')
    
    plt.tight_layout()
    plt.show()

In [ ]:
#####       时段总步数对比柱状图
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

# 定义时间段（保持不变）
periods = {
    '夜间 (0-6h)': (0, 6*3600),
    '上午 (6-12h)': (6*3600, 12*3600),
    '下午 (12-18h)': (12*3600, 18*3600),
    '晚间 (18-24h)': (18*3600, 24*3600)
}

# 计算各时段总步数（保持不变）
period_steps = []
for p_name, (start, end) in periods.items():
    total = 0
    for file in os.listdir("steps"):
        if file.endswith('_steps.txt'):
            df = pd.read_csv(os.path.join("steps", file), 
                            header=None, 
                            names=['seconds', 'steps'])
            mask = (df['seconds'] >= start) & (df['seconds'] < end)
            total += df.loc[mask, 'steps'].sum()
    period_steps.append(total)

# 创建图表（修改单位部分）
plt.figure(figsize=(10,6))
bars = plt.bar(periods.keys(), 
              [s/1000 for s in period_steps],  # 转换为千单位
              color=['#2F4F4F', '#4C72B0', '#55A868', '#C44E52'],
              alpha=0.8)

# 设置标签和标题
plt.title('分时段总步数对比（千单位）', fontsize=14)
plt.ylabel('总步数（千）', fontsize=12)  # 修改单位标签
plt.xticks(rotation=15)

# 添加数值标签（修改格式）
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height*1.02,
            f'{height:,.0f}K',  # 使用千单位格式化
            ha='center', va='bottom',
            fontsize=10)

# 优化网格和布局
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.gca().set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy import signal
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号


def extract_features(acc_data, window_size=60, overlap=0.5):
    # 打印数据基本信息
    print(f"输入数据形状: {acc_data.shape}")
    print(f"数据类型: {acc_data.dtype}")

    # 检查数据是否为空或维度不正确
    if acc_data is None or len(acc_data) == 0:
        print("错误：输入数据为空")
        return np.array([]), []

    # 确保数据是二维的
    if acc_data.ndim == 1:
        acc_data = acc_data.reshape(-1, 1)
    elif acc_data.ndim > 2:
        print(f"错误：不支持的数据维度 {acc_data.ndim}")
        return np.array([]), []

    features = []
    feature_indices = []  # 记录每个特征对应的原始数据索引
    step = max(1, int(window_size * (1 - overlap)))

    # 如果数据长度小于窗口大小，调整窗口大小
    if len(acc_data) < window_size:
        window_size = len(acc_data)

    for i in range(0, max(1, len(acc_data) - window_size + 1), step):
        window = acc_data[i:i+window_size]

        # 时域特征
        try:
            mean = np.mean(window, axis=0)
            std = np.std(window, axis=0)
            max_val = np.max(window, axis=0)
            min_val = np.min(window, axis=0)
        except Exception as e:
            print(f"时域特征提取错误: {e}")
            continue

        # 频域特征 - 更健壮的处理
        try:
            # 对每个轴单独处理
            freq_features = []
            for col in range(window.shape[1]):
                col_data = window[:, col]

                # 如果数据太短，跳过频域分析
                if len(col_data) < 3:
                    freq_features.extend([0, 0])
                    continue

                # 使用更小的 nperseg
                f, Pxx = signal.welch(
                    col_data, fs=1.0, nperseg=min(len(col_data), 10))

                # 安全地处理频率特征
                if len(f) > 0 and len(Pxx) > 0:
                    dominant_freq = f[np.argmax(Pxx)] if len(Pxx) > 0 else 0
                    power_ratio = np.sum(
                        Pxx[f <= 0.5]) / np.sum(Pxx) if np.sum(Pxx) != 0 else 0
                else:
                    dominant_freq = 0
                    power_ratio = 0

                freq_features.extend([dominant_freq, power_ratio])

            # 组合所有特征
            feature_vector = [
                *mean, *std, *max_val, *min_val,
                *freq_features
            ]
            features.append(feature_vector)
            feature_indices.append(i + window_size // 2)  # 记录窗口中心点的索引

        except Exception as e:
            print(f"特征提取错误: {e}")
            continue

    # 返回特征数组和对应的索引
    return np.array(features), feature_indices


def classify_activity(features, feature_indices, total_length):
    if len(features) == 0:
        print("没有可用于分类的特征")
        return np.zeros(total_length, dtype=int), ['无法分类']

    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)

    kmeans = KMeans(n_clusters=4, random_state=42)
    labels = kmeans.fit_predict(scaled_features)

    # 根据聚类中心确定活动强度
    centers = kmeans.cluster_centers_
    activity_levels = ['久坐', '低强度', '中强度', '高强度']

    # 将特征级别的标签映射到原始数据长度
    full_labels = np.zeros(total_length, dtype=int)
    for idx, label_idx in enumerate(labels):
        # 使用特征索引进行插值
        start_idx = feature_indices[idx]
        end_idx = feature_indices[idx+1] if idx + \
            1 < len(feature_indices) else total_length
        full_labels[start_idx:end_idx] = label_idx

    return full_labels, activity_levels


def analyze_weekday_weekend(df):
    df['datetime'] = pd.to_datetime(df['time'])
    df['is_weekend'] = df['datetime'].dt.dayofweek >= 5

    # 创建一个新列，用于更清晰的标签
    df['day_type'] = df['is_weekend'].map({False: '工作日', True: '周末'})

    # 计算每个日期类型和活动强度的比例
    # 使用 crosstab 确保即使某些组合不存在也会包含
    weekday_weekend_analysis = pd.crosstab(
        df['day_type'],
        df['activity_level'],
        normalize='index'
    )

    # 确保所有活动强度都存在，不存在的填充0
    activity_levels = ['久坐', '低强度', '中强度', '高强度']
    for level in activity_levels:
        if level not in weekday_weekend_analysis.columns:
            weekday_weekend_analysis[level] = 0

    # 按指定顺序排序列
    weekday_weekend_analysis = weekday_weekend_analysis.reindex(
        columns=activity_levels)

    return weekday_weekend_analysis


def analyze_user_activity(user_id):
    # 读取数据并筛选特定用户
    df = pd.read_csv('cleaned_sleep_data_fixed_time.csv')
    user_df = df[df['user_id'] == user_id].copy()

    if len(user_df) == 0:
        print(f"未找到用户 {user_id} 的数据")
        return

    print(f"分析用户 {user_id} 的活动模式")
    print(f"用户数据总行数: {len(user_df)}")

    # 提取三轴加速度数据
    acc_data = user_df[['x', 'y', 'z']].values

    # 特征提取
    features, feature_indices = extract_features(acc_data)

    # 如果特征提取失败，打印详细信息
    if len(features) == 0:
        print("警告：未能提取任何特征！")
        return

    # 活动分类
    labels, activity_levels = classify_activity(
        features, feature_indices, len(user_df))
    user_df['activity_level'] = [activity_levels[label] for label in labels]

    # 工作日/周末分析
    weekday_weekend_analysis = analyze_weekday_weekend(user_df)

    # 创建一个更大的图形，以容纳两个子图
    plt.figure(figsize=(20, 10))

    # 活动强度分布 - 第一个子图
    plt.subplot(1, 2, 1)
    sns.countplot(x='activity_level', data=user_df)
    plt.title(f'用户 {user_id} 活动强度分布')
    plt.xlabel('活动强度')
    plt.ylabel('计数')

    # 日常活动强度 - 第二个子图
    plt.subplot(1, 2, 2)
    weekday_weekend_analysis.plot(kind='bar', stacked=False, ax=plt.gca())
    plt.title(f'用户 {user_id} 日常活动强度')
    plt.xlabel('日期类型')
    plt.ylabel('活动强度比例')
    plt.legend(title='活动强度')

    # 调整布局并保存图像
    plt.tight_layout()
    plt.savefig(f'user_{user_id}_activity_analysis_plots.png',
                bbox_inches='tight')

    # 保存结果
    user_df.to_csv(
        f'user_{user_id}_activity_analysis_results.csv', index=False)

    # 打印分析结果，帮助诊断
    print(f"用户 {user_id} 日常活动强度分析结果：")
    print(weekday_weekend_analysis)

    return user_df, weekday_weekend_analysis

# 主函数


def main():
    # 分析特定用户
    analyze_user_activity(1066528)


if __name__ == "__main__":
    main()